In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS e_comm_databricks.silver;


In [0]:
df_orders = spark.table("e_comm_databricks.bronze.olist_orders")

In [0]:
display(df_orders)

In [0]:
df_orders.count()

In [0]:
df_orders.printSchema()

In [0]:
df_orders = spark.table("e_comm_databricks.bronze.olist_orders")

In [0]:
df_orders.groupBy("order_id").count().filter("count > 1").show()

In [0]:
df_orders_clean = df_orders.dropDuplicates(["order_id"])

In [0]:
df_orders_clean.count()

In [0]:
df_orders_clean.filter("Order_id IS NULL").display()

In [0]:
df_orders_clean = df_orders_clean.dropna(subset =["order_id"])

In [0]:
df_orders_clean.select("order_status").distinct().show()

In [0]:
df_orders_clean.filter("order_status = 'unavailable'").display()

In [0]:
df_orders_clean.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("e_comm_databricks.silver.orders")

Customers Table

In [0]:
df_customers = spark.table("e_comm_databricks.bronze.olist_customers")

In [0]:
df_customers.display()

In [0]:
df_customers.groupBy("customer_unique_id").count().filter("count > 1").show()

In [0]:
df_customers_clean = df_customers.dropDuplicates(["customer_unique_id"])

In [0]:
df_customers_clean.groupBy("customer_unique_id").count().filter("count>1").display()

In [0]:
df_customers_clean.groupBy("customer_id").count().filter("count>1").display()

In [0]:
df_customers_clean.filter("customer_id IS NULL").count()

In [0]:
df_customers_clean.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("e_comm_databricks.silver.customers")

In [0]:
df_customers_clean.printSchema()

Products Table

In [0]:
df_products = spark.table("e_comm_databricks.bronze.olist_products")

In [0]:
df_products.display()

In [0]:
df_products.groupBy("product_id").count().filter("count>1").display()

In [0]:
df_products.filter("product_id IS NULL").count()

In [0]:
df_products_clean = df_products.dropDuplicates(["product_id"])

In [0]:
df_products_clean = df_orders_clean.dropna(subset=["product_id"])


In [0]:
df_products_clean.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("e_comm_databricks.silver.products")

In [0]:
df_products_clean.printSchema()

In [0]:
df_sellers = spark.table("e_comm_databricks.bronze.olist_sellers")
df_sellers.display()

In [0]:
df_sellers.groupBy("seller_id").count().filter("count>1").display()

In [0]:
df_sellers_clean = df_sellers.dropDuplicates(["seller_id"])

In [0]:
df_sellers_clean.filter("seller_id IS NULL").count()

In [0]:
df_sellers_clean.dropna(subset = ["seller_id"])

In [0]:
df_sellers_clean.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("e_comm_databricks.silver.sellers")

In [0]:
df_product_category_trans = spark.table("e_comm_databricks.bronze.olist_product_category_translation")

In [0]:
df_product_category_trans.display()

In [0]:
df_product_category_trans_clean = df_product_category_trans.dropDuplicates(["product_category_name"])


In [0]:
df_product_category_trans_clean.display()

In [0]:
df_product_category_trans_clean.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("e_comm_databricks.silver.product_category_translation")


Geolocation

In [0]:
df_geo = spark.table("e_comm_databricks.bronze.olist_geolocation")

In [0]:
df_geo.display()

In [0]:
df_geo = df_geo.groupBy("geolocation_zip_code_prefix").count().filter("count>1").display()

In [0]:
df_geo_clean = df_geo.dropDuplicates(['geolocation_zip_code_prefix'])

In [0]:
df_geo_clean.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("e_comm_databricks.silver.geolocation")

Order_items

In [0]:
df_order_items = spark.table("e_comm_databricks.bronze.olist_order_items")

In [0]:
df_order_items.display()

In [0]:
df_order_items.groupBy("order_id","order_item_id").count().filter("count>1").display()

In [0]:
df_order_items_clean = df_order_items.dropDuplicates(['order_id','order_item_id'])

In [0]:
df_order_items_clean.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("e_comm_databricks.silver.order_items")

Order_payments

In [0]:
df_order_payments = spark.table("e_comm_databricks.bronze.olist_order_payments")

In [0]:
df_order_payments.display()

In [0]:
df_order_payments.groupBy("order_id","payment_sequential").count().filter("count>1").display()

In [0]:
df_order_payments_clean = df_order_payments.dropDuplicates(["order_id", "payment_sequential"])

In [0]:
df_order_payments_clean.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("e_comm_databricks.silver.order_payments")

order Reviews

In [0]:
df_order_reviews =spark.table("e_comm_databricks.bronze.olist_order_reviews")

In [0]:
df_order_reviews.display()

In [0]:
df_order_reviews.groupBy("review_id").count().filter("count>1").show()

In [0]:
df_reviews_clean = df_order_reviews.dropDuplicates(["review_id"])

In [0]:
df_reviews_clean.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("e_comm_databricks.silver.order_reviews")


Orders ↔ Customers Relationship

In [0]:
df_orders = spark.table("e_comm_databricks.silver.orders")
df_customers = spark.table("e_comm_databricks.silver.customers")

In [0]:
df_orders.join(
    df_customers,
    on="customer_id",
    how="left_anti"
).display()


order_items ↔ products

In [0]:
df_items = spark.table("e_comm_databricks.silver.order_items")
df_products = spark.table("e_comm_databricks.silver.products")


In [0]:
df_items.join(df_products, on="product_id",how="left_anti").display()


order_items ↔ sellers

In [0]:
df_sellers = spark.table("e_comm_databricks.silver.sellers")


In [0]:
df_items.join(
    df_sellers,
    on="seller_id",
    how="left_anti"
).display()


When you see unexpected rows in left_anti join, that is exactly where real data engineering thinking begins.

You ran:

df_orders LEFT_ANTI JOIN df_customers ON customer_id


and got:

👉 3000+ rows

Meaning:

orders exist whose customer_id does NOT exist in customers table


Let’s understand WHY this happens.

🚀 First — What left_anti means
left_anti


returns:

👉 rows from LEFT table that have NO match in RIGHT table.

So:

orders.customer_id NOT IN customers.customer_id

🧠 Possible reasons (real-world scenarios)
✅ 1 — Deduplication removed valid customer records

Earlier in Silver you did:

dropDuplicates(...)


If dedup logic wrong:

👉 valid customers may have been removed.

Example:

If you dedup using wrong column.

✅ 2 — Null values in customer_id

Check:

df_orders.filter("customer_id IS NULL").count()


Nulls will appear in left_anti.

✅ 3 — Data ingestion mismatch

Maybe:

customers bronze table not fully loaded.

customers silver table missing rows.

✅ 4 — Composite logic misunderstood

In Olist dataset:

Important detail:

customer_id ≠ customer_unique_id


Sometimes people mistakenly use wrong column.

✅ 5 — Data quality issue in source dataset

Yes — sometimes raw datasets contain orphan records.

Real engineers must validate.

In [0]:
df_orders.select("customer_id").distinct().count()
df_customers.select("customer_id").distinct().count()


In [0]:
df_orders.join(
    df_customers,
    on="customer_id",
    how="left_anti"
).select("customer_id").distinct().display(20, False)


In [0]:
df_orders.join(
    df_customers,
    on="customer_id",
    how="left_anti"
)


In [0]:
sample_id = "b5c1c56fe1ec3cec6893b98d90a339bd"
from pyspark.sql.functions import col
df_customers.filter(col("customer_id") == sample_id).display()


In [0]:
df_orders.select("customer_id").distinct().count()

df_customers.select("customer_id").distinct().count()


In [0]:
missing_count = df_orders.join(
    df_customers,
    on="customer_id",
    how="left_anti"
).count()


In [0]:
df_orders.join(df_customers, "customer_id") \
         .select("order_id", "customer_city", "customer_state") \
         .show()


In [0]:
valid_orders = df_orders.join(df_customers, "customer_id", "inner")

In [0]:
df_orders = spark.table("e_comm_databricks.silver.orders")
df_customers = spark.table("e_comm_databricks.silver.customers")


In [0]:
valid_orders = df_orders.join(
    df_customers.select("customer_id"),
    on="customer_id",
    how="inner"
)


In [0]:
invalid_orders = df_orders.join(
    df_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
)


In [0]:
valid_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_comm_databricks.silver.orders")


In [0]:
invalid_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_comm_databricks.silver.orders_quarantine")
